# Fig. 4: Squeezing spectrum vs. cavity decay (effective model)

Effective lasing model with the same parameters and $\kappa$ grid as `3_full_model_kappa.ipynb`: squeezing spectrum (**lines of panel d**) and positions of the two spectral lines (**lines of panel e**).

**Outputs** (`../data/kappa/`): `Ssq_eff.npy`, `eff_positions.npy`.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using LinearAlgebra
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_k = 100

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = 5000
N_atom_II = 300
N_atom = N_atom_I .+ N_atom_II

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = logrange(0.01,20,num_k)*2*pi*1e-3 # Cavity decay, scanned
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.8 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

P = 1e-4
P_T = P.+P_eff

C = (2*N_atom_II.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

purc = 4*g_02./(dec_f.+kappa)

## Cumulant expansion of the effective model

In [ ]:
@rnumbers Δ λ_r λ_s κ κ_f R N p γ_ϕ γ # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of the cavity
h_atom = NLevelSpace(:atom,(:g,:e)) # Hilbert space of the atom
h = h_cav⊗h_atom # Complete Hilbert space

as = Destroy(h,:as)
a = as*cosh(R)-as'*sinh(R)
σ(x,y,k) = IndexedOperator(Transition(h, :σ, x, y), k) # Operators

i = Index(h, :i, N, h_atom)
j = Index(h, :j, N, h_atom)

H = -Δ*as'*as + ∑(λ_r*(a*σ(:e,:g,i)+a'*σ(:g,:e,i)) + λ_s*(a'*σ(:e,:g,i)+a*σ(:g,:e,i)),i)

D = [a;as;σ(:e,:g,i);σ(:e,:e,i)-σ(:g,:g,i);σ(:g,:e,i)]
rates = [κ; κ_f; p; γ_ϕ; γ]

ops = [as] # Operators whose equations we derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Second-order cumulant expansion
me_comp = complete!(eqs) # Complete the set of equations
eqs_sc = scale(me_comp) # Use the permutation symmetry of the atoms

corr = CorrelationFunction(as',as,me_comp; steady_state=true)
corr2 = CorrelationFunction(as,as',me_comp; steady_state=true)
corr3 = CorrelationFunction(as,as,me_comp; steady_state=true)
corr4 = CorrelationFunction(as',as',me_comp; steady_state=true)

corr_sc = scale(corr)
corr_sc2 = scale(corr2)
corr_sc3 = scale(corr3)
corr_sc4 = scale(corr4)

@named sys = System(eqs_sc)

param_list = (Δ, λ_r, λ_s, κ, κ_f, R, N, p, γ_ϕ, γ)
S = Spectrum(corr_sc,param_list)
S2 = Spectrum(corr_sc2,param_list)
S3 = Spectrum(corr_sc3,param_list)
S4 = Spectrum(corr_sc4,param_list)

## Steady state and spectra

Each point is integrated repeatedly (up to 10 times) until $\langle a^\dagger a\rangle$ and $\langle a^2\rangle$ stop changing over the integration window, re-using the previous solution as initial condition; the time-averaged state over one more window is used for the spectra.

In [ ]:
ada = zeros(num_k) # <a_s^† a_s>
sda = zeros(ComplexF64,num_k) # <σ^† a_s>
M = zeros(num_k) # Excited-state population of one gain atom

freqs = LinRange(-2,2,2000)*1e-3*2pi
spec = zeros(num_k,length(freqs)) # Squeezing spectrum

it = zeros(num_k)
dist = zeros(num_k)
dist2 = zeros(num_k)

progress = Progress(num_k)
safe_lock = ReentrantLock()

Threads.@threads for l in 1:num_k

    u0 = zeros(ComplexF64, length(eqs_sc))
    sol = nothing
    p0 = nothing
    dict = nothing
    prob = nothing

    dist[l] = 10
    dist2[l] = 1

    while (dist[l]>1 || dist2[l]>0.01) && it[l] < 10

        p0 = (delta_lamb_f, lambda_r, lambda_s, kappa[l], dec_f, r, N_atom_II, P_T, dep_T, dec_T)
        dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

        prob = ODEProblem(sys,dict,(0.0,30000.0))
        sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

        lock(safe_lock) do
            dist_new = maximum(abs.(sol[as'as]))-minimum(abs.(sol[as'as]))
            dist2[l] = maximum(abs.(sol[as*as]))-minimum(abs.(sol[as*as]))

            if abs(dist_new - dist[l])/dist[l]<0.02 && dist[l] > 10
                dist[l] = dist_new
                it[l] = 10
            else
                dist[l] = dist_new
            end
        end

        u0 = sol.u[end]

        it[l] = it[l]+1

    end

    if it[l] <= 10

        prob = ODEProblem(sys,dict,(0.0,30000.0))
        sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

        u0 = sol.u[end]

        n_t = length(sol.t)
        u_avg = sum(sol.u)/n_t # Time-averaged state

        # Spectra of the bare mode a = a_s cosh r - a_s^† sinh r
        emis_spec = cosh(r)^2*S(freqs,u_avg,p0) .+ sinh(r)^2*S2(freqs,u_avg,p0).-cosh(r)*sinh(r)*(S3(freqs,u_avg,p0) .+ S4(freqs,u_avg,p0))
        anom_spec = cosh(r)^2*S3(freqs,u_avg,p0) .+ sinh(r)^2*S4(freqs,u_avg,p0).-cosh(r)*sinh(r)*(S(freqs,u_avg,p0) .+ S2(freqs,u_avg,p0))
        emis_spec_inv = cosh(r)^2*S(-freqs,u_avg,p0) .+ sinh(r)^2*S2(-freqs,u_avg,p0).-cosh(r)*sinh(r)*(S3(-freqs,u_avg,p0) .+ S4(-freqs,u_avg,p0))
        anom_spec_inv = cosh(r)^2*S3(-freqs,u_avg,p0) .+ sinh(r)^2*S4(-freqs,u_avg,p0).-cosh(r)*sinh(r)*(S(-freqs,u_avg,p0) .+ S2(-freqs,u_avg,p0))

        spec[l,:] =real.(1 .+ kappa[l]*(emis_spec .+ emis_spec_inv .+ anom_spec*exp(2*im*0) .+ anom_spec_inv*exp(-2*im*0)))

        lock(safe_lock) do
            ada[l] = sum(real.(sol[as'*as]))/n_t
            sda[l] = sum(sol[σ(:e,:g,1)*as])/n_t
            M[l] = sum(real.(sol[σ(:e,:e,1)]))/n_t
        end
    end
    next!(progress)
end

## Spectral lines

Eigenvalues of the linearised equations of the correlation functions: imaginary parts give the line positions.

In [ ]:
function spec_matrix(κ,g_0,N,ω_c,P,ϕ,γ,sz) # Linearised equations of <a_s^† a_s(τ)> and <σ^† a_s(τ)>

    M = zeros(ComplexF64,2,2)
    Γ = P+γ+4*ϕ

    M[1,1]=im*ω_c-κ/2
    M[1,2]=im*g_0*N

    M[2,1]=-im*g_0*sz
    M[2,2]=-Γ/2

    return M
end

Γ = zeros(num_k,2) # Widths of the two lines
ω = zeros(num_k,2) # Positions of the two lines

for l in 1:num_k
    F = eigen(conj.(spec_matrix(kappa[l]+dec_f, sqrt.(g_02), N_atom_II, -delta_lamb_f, P_T, dep_T, dec_T, 2*M[l]-1)))
    Γ[l,:] = -2*real.(F.values)
    ω[l,:] = imag(F.values)
end

## Save

In [ ]:
sq_db = 20*log10.(sqrt.(spec)) # 10 log10(S) in dB: negative values mean squeezing

data_dir = joinpath("..", "data", "kappa") # Relative to this notebook

npzwrite(joinpath(data_dir, "Ssq_eff.npy"), sq_db)
npzwrite(joinpath(data_dir, "eff_positions.npy"), ω)